## Vdelave

V našem prejšnjem primeru smo delovali na visokodimenzionalnih vektorjih vrečke besed z dolžino `vocab_size` in smo eksplicitno pretvarjali nizkodimenzionalne vektorje pozicijskega predstavitve v redko predstavitev z eno vročo kodirko. Ta ena vroča predstavitev ni pomnilniško učinkovita, poleg tega je vsaka beseda obravnavana neodvisno od drugih, tj. ena vroče kodirani vektorji ne izražajo nobene semantične podobnosti med besedami.

V tej enoti bomo nadaljevali z raziskovanjem **News AG** nabora podatkov. Za začetek naložimo podatke in pridobimo nekaj definicij iz prejšnjega zvezka.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Kaj je vdelava (embedding)?

Ideja **vdelave** (embedding) je predstaviti besede z nižjimi dimenzijskimi gostimi vektorji, ki nekako odražajo semantični pomen besede. Kasneje bomo razpravljali, kako zgraditi smiselne vdelave besed, za zdaj pa si vdelave predstavljajmo kot način znižanja dimenzionalnosti vektorja besede.

Torej bo vdelavna plast vzela besedo kot vhod in dala izhodni vektor določene `embedding_size`. V nekem smislu je zelo podobna plasti `Linear`, vendar namesto enoglasbeno kodiranega vektorja sprejme številko besede kot vhod.

Z uporabo vdelavne plasti kot prve plasti v naši mreži lahko preklopimo iz modela vrečke besed v model **embedding bag**, kjer najprej pretvorimo vsako besedo v besedilu v ustrezno vdelavo, nato pa za vse te vdelave izračunamo neko agregatno funkcijo, kot so `sum`, `average` ali `max`.

![Slika, ki prikazuje vdelavni razvrščalnik za pet zaporednih besed.](../../../../../translated_images/sl/embedding-classifier-example.b77f021a7ee67eee.webp)

Naša nevronska mreža za razvrščanje se bo začela z vdelavno plastjo, nato z agregatno plastjo in linearno klasifikacijo na vrhu:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Ravnanje z variabilno dolžino zaporedij

Zaradi te arhitekture bi bilo treba minibatche za našo mrežo ustvarjati na določen način. V prejšnji enoti, ko smo uporabljali bag-of-words, so imeli vsi BoW tenzorji v minibatchu enako velikost `vocab_size`, ne glede na dejansko dolžino naše besedilne sekvence. Ko preidemo na vdelavo besed, bi imeli spremenljivo število besed v vsakem besedilnem vzorcu, in ko bi te vzorce združevali v minibatche, bi morali uporabiti nekakšno polnjenje.

To lahko storimo z uporabo iste tehnike zagotavljanja funkcije `collate_fn` za podatkovni vir:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Usposabljanje razvrščevalnika vdelav

Zdaj, ko smo definirali ustrezen nalagalnik podatkov, lahko model usposobimo z uporabo funkcije za usposabljanje, ki smo jo definirali v prejšnji enoti:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Opomba**: Tukaj treniramo le na 25 tisoč zapisih (manj kot en celoten epoh) zaradi varčevanja s časom, vendar lahko nadaljujete s treniranjem, napišete funkcijo za treniranje več epoh in eksperimentirate z nastavkom hitrosti učenja, da dosežete višjo natančnost. Natančnost bi morala doseči približno 90 %.


### Plast EmbeddingBag in predstavitev zaporedij spremenljive dolžine

V prejšnji arhitekturi smo vse zaporedja morali dopolniti na enako dolžino, da smo jih lahko združili v minibatch. To ni najbolj učinkovit način za predstavitev zaporedij spremenljive dolžine - druga možnost bi bila uporaba **offset** vektorja, ki bi hranil offsete vseh zaporedij shranjenih v enem velikem vektorju.

![Slika, ki prikazuje predstavitev zaporedja z offsetom](../../../../../translated_images/sl/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Opomba**: Na zgornji sliki prikazujemo zaporedje znakov, a v našem primeru delamo z zaporedji besed. Vendar pa splošno načelo predstavitve zaporedij z offset vektorjem ostaja enako.

Za delo s predstavitvijo z offsetom uporabljamo plast [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Ta je podobna `Embedding`, vendar kot vhod vzame vsebinski vektor in offset vektor, vključuje pa tudi povprečno plast, ki je lahko `mean`, `sum` ali `max`.

Tukaj je spremenjena mreža, ki uporablja `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Za pripravo nabora podatkov za učenje moramo zagotoviti funkcijo pretvorbe, ki bo pripravila premik vektorja:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Upoštevajte, da naša mreža, za razliko od vseh prejšnjih primerov, zdaj sprejema dva parametra: vektor podatkov in vektor zamika, ki sta različnih velikosti. Prav tako nam naš nalagalec podatkov zagotavlja 3 vrednosti namesto 2: tako vektorji besedila kot zamika so na voljo kot značilnosti. Zato moramo nekoliko prilagoditi našo funkcijo učenja, da se s tem ustrezno spoprime:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantične vdelave: Word2Vec

V našem prejšnjem primeru se je sloj modela vdelave naučil preslikati besede v vektorsko reprezentacijo, vendar ta reprezentacija ni imela veliko semantičnega pomena. Bilo bi dobro naučiti se takšne vektorske reprezentacije, kjer bi podobne besede ali sopomenke ustrezale vektorjem, ki so si med seboj blizu glede na neko vektorsko razdaljo (npr. evklidska razdalja).

Za to moramo naš vdelavni model predhodno trenirati na veliki zbirki besedil na poseben način. Eden prvih načinov za učenje semantičnih vdelav je imenovan [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Temelji na dveh glavnih arhitekturah, ki se uporabljata za izdelavo distribuirane reprezentacije besed:

 - **Neprekinjena vrečka besed** (CBoW) — v tej arhitekturi treniramo model, da napove besedo iz okoliškega konteksta. Glede na n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$ je cilj modela napovedati $W_0$ iz $(W_{-2},W_{-1},W_1,W_2)$.
 - **Neprekinjen skip-gram** je nasprotje CBoW. Model uporablja okolišno okno kontekstnih besed za napoved trenutne besede.

CBoW je hitrejši, medtem ko je skip-gram počasnejši, a bolje predstavi redke besede.

![Slika prikazuje algoritma CBoW in Skip-Gram za pretvorbo besed v vektorje.](../../../../../translated_images/sl/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Za eksperimentiranje z word2vec vdelavo, predhodno trenirano na Google News naboru podatkov, lahko uporabimo knjižnico **gensim**. Spodaj najdemo besede, ki so najbolj podobne 'neural'

> **Opomba:** Ko prvič ustvarjate vektorje besed, lahko prenos traja nekaj časa!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Iz besede lahko tudi izračunamo vektorske vdelave, ki se uporabijo pri usposabljanju klasifikacijskega modela (za jasnost pokažemo le prvih 20 komponent vektorja):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Odlično pri semantičnih vdelavah je, da lahko manipulirate z vektorsko kodiranjem, da spremenite semantiko. Na primer, lahko vprašamo za iskanje besede, katere vektorska predstavitev bi bila čim bližje besedam *king* in *woman*, ter čim dlje od besede *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Tako CBoW kot Skip-Grams sta "napovedni" vdelavi, saj upoštevata samo lokalne kontekste. Word2Vec ne izkorišča globalnega konteksta. 

**FastText** nadgrajuje Word2Vec z učenjem vektorskih predstavitev za vsako besedo in znakovne n-grame znotraj vsake besede. Vrednosti teh predstavitev se nato na vsakem koraku učenja povprečijo v en vektor. Čeprav to doda veliko dodatnih izračunov v prerazporejanju, omogoča, da vdelave besed kodirajo informacije o podbesedah. 

Drug pristop, **GloVe**, izkorišča idejo matrike so-ponavadi, uporablja nevronske metode za razčlenitev matrike so-ponavadi v bolj izražajne in nelinearne vektorske predstavitve besed.

Z vzorcem lahko eksperimentirate z zamenjavo vdelav z FastText in GloVe, saj gensim podpira več različnih modelov za vdelave besed.


## Uporaba vnaprej naučenih vdelav v PyTorch

Primer zgoraj lahko spremenimo tako, da v matriko plasti vdelave vnaprej naložimo semantične vdelave, kot je Word2Vec. Upoštevati moramo, da se besednjaki vnaprej naučenih vdelav in našega besedilnega korpusa verjetno ne bodo ujemali, zato bomo uteži za manjkajoče besede inicializirali z naključnimi vrednostmi:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Zdaj pa trenirajmo naš model. Upoštevajte, da je čas, potreben za treniranje modela, občutno daljši kot v prejšnjem primeru, zaradi večje velikosti plasti za vdelavo in s tem veliko večjega števila parametrov. Prav tako bomo morda morali model trenirati na več primerih, če želimo preprečiti prekomerno prileganje.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

V našem primeru ne opazimo velikega povečanja natančnosti, kar je verjetno posledica precej različnih besednjakov. 
Da bi premagali težavo različnih besednjakov, lahko uporabimo eno od naslednjih rešitev:
* Ponovno usposobimo word2vec model na našem besednjaku
* Naložimo naš nabor podatkov z besednjakom iz vnaprej usposobljenega word2vec modela. Besednjak, uporabljen za nalaganje nabora podatkov, je mogoče določiti med nalaganjem.

Drugi pristop se zdi lažji, še posebej zato, ker PyTorch `torchtext` ogrodje vsebuje vgrajeno podporo za vdelave. Na primer, slovar na osnovi GloVe lahko ustvarimo na naslednji način:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Naložen slovar ima naslednje osnovne operacije:
* `vocab.stoi` slovar nam omogoča pretvorbo besede v njen indeks v slovarju
* `vocab.itos` naredi nasprotno - pretvori številko v besedo
* `vocab.vectors` je polje vektorskih upodobitev, zato za pridobitev upodobitve besede `s` moramo uporabiti `vocab.vectors[vocab.stoi[s]]`

Tukaj je primer manipulacije z upodobitvami za demonstracijo enačbe **kind-man+woman = queen** (moral sem malo prilagoditi koeficient, da je delovalo):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Da treniramo klasifikator z uporabo teh vdelav, moramo najprej kodirati naš nabor podatkov z uporabo besedišča GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Kot smo videli zgoraj, so vse vektorske vdelave shranjene v matriki `vocab.vectors`. To zelo olajša nalaganje teh uteži v uteži plasti vdelave z uporabo preprostega kopiranja:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Zdaj izurimo naš model in preverimo, če dobimo boljše rezultate:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Eden od razlogov, da ne vidimo bistvenega povečanja natančnosti, je dejstvo, da nekaj besed iz našega nabora podatkov manjka v vnaprej naučenem slovarju GloVe, zato jih v bistvu ignoriramo. Da bi premagali ta problem, lahko na našem naboru podatkov treniramo lastne vgradnje. 


## Kontekstualne vgradnje

Ena ključnih omejitev tradicionalnih vnaprej naučenih predstavitev vgradnje, kot je Word2Vec, je težava razločevanja pomena besed. Medtem ko lahko vnaprej naučene vgradnje zajamejo del pomena besed v kontekstu, je vsak možni pomen besede kodiran v isto vgradnjo. To lahko povzroči težave v nadaljnjih modelih, saj ima veliko besed, na primer beseda 'play', različne pomene glede na kontekst, v katerem se uporabljajo.

Na primer, beseda 'play' v teh dveh različnih stavkih ima precej drugačen pomen:
- Šel sem na **igro** v gledališče.
- John želi **igrati** s svojimi prijatelji.

Zgornje vnaprej naučene vgradnje predstavljajo oba pomena besede 'play' v isti vgradnji. Da bi premagali to omejitev, moramo izdelati vgradnje, ki temeljijo na **jezikovnem modelu**, ki je usposobljen na velikem korpusu besedila in *ve*, kako je mogoče besede združiti v različnih kontekstih. Razprava o kontekstualnih vgradnjah je izven obsega tega vodiča, vendar se bomo nanje vrnili pri pogovoru o jezikovnih modelih v naslednji enoti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
